In [ ]:
!pip install requests beautifulsoup4 pandas lxml tqdm --quiet

import requests, time, random, os, re, json, logging
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import urllib.robotparser
from collections import deque
from io import StringIO
import pandas as pd
from tqdm import tqdm

START_URL = "https://www.worldometers.info/"
ALLOWED_DOMAIN = urlparse(START_URL).netloc
OUTPUT_DIR = "/content/worldometers_full_csvs"
VISITED_FILE = os.path.join(OUTPUT_DIR, "visited.json")
INDEX_FILE = os.path.join(OUTPUT_DIR, "index.csv")
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; MyCrawler/1.0; +https://example.com/bot)"}

# Crawl behavior (اضبط إذا أردت)
MAX_PAGES = None               # None = لا حد؛ أو ضع رقمًا لعدد صفحات للاختبار
REQUEST_DELAY = (1.0, 3.0)     # فترة انتظار عشوائية بين الطلبات (ثواني)
RETRIES = 3
TIMEOUT = 15

# ====== Helpers ======
def safe_filename(s):
    s = re.sub(r"https?://", "", s)
    s = re.sub(r"[^\w\-_. ]", "_", s)
    s = re.sub(r"\s+", "_", s).strip("_")
    return s[:200]

def make_dirs():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_visited():
    if os.path.exists(VISITED_FILE):
        try:
            with open(VISITED_FILE, "r", encoding="utf-8") as f:
                return set(json.load(f))
        except:
            return set()
    return set()

def save_visited(visited):
    with open(VISITED_FILE, "w", encoding="utf-8") as f:
        json.dump(list(visited), f, ensure_ascii=False, indent=2)

def is_same_domain(url):
    parsed = urlparse(url)
    return parsed.netloc == ALLOWED_DOMAIN or parsed.netloc.endswith("."+ALLOWED_DOMAIN)

def normalize_url(base, link):
    if not link:
        return None
    if link.startswith("javascript:") or link.startswith("mailto:") or link.startswith("tel:") or link.startswith("#"):
        return None
    return urljoin(base, link.split("#")[0])

# ====== robots.txt check ======
def can_fetch(url, user_agent=HEADERS["User-Agent"]):
    robots_url = urljoin(START_URL, "/robots.txt")
    rp = urllib.robotparser.RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # إذا فشل جلب robots.txt نتبع سلوك محافظ (اسمح)
        return True

# ====== Fetch page safely ======
def fetch(url):
    for attempt in range(RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            return r.text
        except Exception as e:
            wait = (2 ** attempt) + random.random()
            time.sleep(wait)
    return None

# ====== Extract texts (headings, paragraphs, list items, spans) and save CSV ======
def extract_and_save_texts(url, html):
    soup = BeautifulSoup(html, "lxml")
    page_path = urlparse(url).path or "root"
    page_name = safe_filename(page_path)

    # اجمع عناصر نصية مفيدة
    elements = []
    # headings
    for h in soup.find_all(["h1","h2","h3","h4","h5","h6"]):
        text = h.get_text(separator=" ", strip=True)
        if text:
            elements.append({"__source_url": url, "__page": page_name, "element_type": h.name, "content": text})
    # paragraphs
    for p in soup.find_all("p"):
        text = p.get_text(separator=" ", strip=True)
        if text:
            elements.append({"__source_url": url, "__page": page_name, "element_type": "p", "content": text})
    # list items
    for li in soup.find_all("li"):
        text = li.get_text(separator=" ", strip=True)
        if text:
            elements.append({"__source_url": url, "__page": page_name, "element_type": "li", "content": text})
    # some spans (short texts)
    for sp in soup.find_all("span"):
        text = sp.get_text(separator=" ", strip=True)
        if text and len(text) > 1 and len(text) < 500:
            elements.append({"__source_url": url, "__page": page_name, "element_type": "span", "content": text})

    # Meta / title
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    if title:
        elements.insert(0, {"__source_url": url, "__page": page_name, "element_type": "title", "content": title})

    # حفظ CSV نصوص الصفحة
    if elements:
        df_texts = pd.DataFrame(elements)
        texts_filename = f"{page_name}_texts.csv"
        texts_path = os.path.join(OUTPUT_DIR, texts_filename)
        try:
            df_texts.to_csv(texts_path, index=False, encoding="utf-8-sig")
            return texts_path
        except Exception as e:
            logging.warning(f"فشل حفظ نصوص {url}: {e}")
            return None
    return None

# ====== Extract tables and save CSVs (uses StringIO to avoid FutureWarning) ======
def extract_and_save_tables(url, html):
    page_path = urlparse(url).path or "root"
    page_name = safe_filename(page_path)
    saved_files = []
    try:
        # استخدم pd.read_html على StringIO لتفادي FutureWarning
        dfs = pd.read_html(StringIO(html))
    except Exception:
        dfs = []  # سنحاول باستخراج يدوي أدناه

    if not dfs:
        soup = BeautifulSoup(html, "lxml")
        tables = soup.find_all("table")
        for table in tables:
            headers = [th.get_text(strip=True) for th in table.find_all("th")]
            rows = []
            for tr in table.find_all("tr"):
                cells = tr.find_all(["td","th"])
                row = [cell.get_text(strip=True) for cell in cells]
                if any(cell.strip() for cell in row):
                    rows.append(row)
            if rows:
                if not headers or len(headers) != len(rows[0]):
                    headers = [f"col_{i}" for i in range(1, max(len(r) for r in rows)+1)]
                df_temp = pd.DataFrame(rows, columns=headers[:len(rows[0])])
                dfs.append(df_temp)

    # حفظ DataFrames كـ CSV منفصل لكل جدول
    for i, df in enumerate(dfs, start=1):
        df_insert = df.copy()
        df_insert["__source_url"] = url
        df_insert["__page"] = page_name
        df_insert["__table_index"] = i
        filename = f"{page_name}_table_{i}.csv"
        filepath = os.path.join(OUTPUT_DIR, filename)
        try:
            df_insert.to_csv(filepath, index=False, encoding="utf-8-sig")
            saved_files.append(filepath)
        except Exception as e:
            logging.warning(f"فشل حفظ {filepath}: {e}")
    return saved_files

# ====== Crawler main ======
def crawl(start_url):
    make_dirs()
    visited = load_visited()
    index_records = []
    q = deque([start_url])
    pages_processed = 0

    # robots.txt check
    if not can_fetch(start_url):
        print("robots.txt يمنع الزحف على هذا الموقع — إلغاء.")
        return

    pbar = tqdm(total=MAX_PAGES or 0, desc="Pages (approx)") if MAX_PAGES else None

    while q:
        url = q.popleft()
        if url in visited:
            continue
        if not is_same_domain(url):
            continue

        if not can_fetch(url):
            logging.info(f"ممنوع جلب {url} حسب robots.txt")
            visited.add(url)
            save_visited(visited)
            continue

        html = fetch(url)
        if html is None:
            logging.warning(f"فشل تحميل: {url}")
            visited.add(url)
            save_visited(visited)
            continue

        # حفظ النصوص
        texts_path = extract_and_save_texts(url, html)
        table_paths = extract_and_save_tables(url, html)

        index_records.append({
            "url": url,
            "texts_csv": texts_path,
            "tables_count": len(table_paths)
        })

        soup = BeautifulSoup(html, "lxml")
        for a in soup.find_all("a", href=True):
            normalized = normalize_url(url, a['href'])
            if normalized and is_same_domain(normalized):
                if re.search(r"\.(pdf|zip|rar|7z|mp4|mp3|jpg|png|gif)$", normalized, re.IGNORECASE):
                    continue
                if normalized not in visited:
                    q.append(normalized)

        visited.add(url)
        save_visited(visited)

        pages_processed += 1
        if pbar:
            pbar.update(1)
        if MAX_PAGES and pages_processed >= MAX_PAGES:
            break

        time.sleep(random.uniform(*REQUEST_DELAY))

    idx_df = pd.DataFrame(index_records)
    if not idx_df.empty:
        idx_df.to_csv(INDEX_FILE, index=False, encoding="utf-8-sig")

    if pbar:
        pbar.close()
    print(f"➡ تم معالجة {pages_processed} صفحة. فهرس محفوظ في: {INDEX_FILE}")

if __name__ == "__main__":
    print("ابدأ الزحف — حفظ نصوص + جداول لكل صفحة في:", OUTPUT_DIR)
    crawl(START_URL)
